In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/srv/johan/Python/E9 E10 Data/E9E10 Neural/E9E10_CNS_ONLY.h5ad")
print(adata)
print("\nMetadata columns available:")
print(adata.obs.columns.tolist())

In [ ]:
# Replace R's NA sentinel with the string "NA" that sclitr expects
adata.obs["cloneid"] = adata.obs["cloneid"].replace(-2147483648, "NA").astype(str)

print("After conversion:")
print(adata.obs["cloneid"].head(10))
print("NA count:", (adata.obs["cloneid"] == "NA").sum())
print("Barcoded count:", (adata.obs["cloneid"] != "NA").sum())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Clone size distribution ───────────────────────────────────────────────────
clone_sizes = adata.obs["cloneid"].value_counts()
clone_sizes = clone_sizes[clone_sizes.index != "NA"]

print("── Clone size distribution ──────────────────────────────────────────")
print(f"  {'Min size':>10s}  {'Clones kept':>12s}  {'Cells kept':>12s}  {'% of barcoded':>15s}")
for min_s in [2, 3, 4, 5, 6, 7, 8, 10, 15, 20]:
    clones_kept = (clone_sizes >= min_s).sum()
    cells_kept  = clone_sizes[clone_sizes >= min_s].sum()
    pct         = cells_kept / clone_sizes.sum() * 100
    print(f"  {min_s:>10d}  {clones_kept:>12d}  {cells_kept:>12d}  {pct:>14.1f}%")

# ── Distribution plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution (capped at 30 for readability)
size_counts = clone_sizes.value_counts().sort_index()
size_counts_capped = size_counts[size_counts.index <= 30]
axes[0].bar(size_counts_capped.index, size_counts_capped.values,
            color="#534AB7", alpha=0.8, width=0.8)
axes[0].set_xlabel("Clone size (number of cells)")
axes[0].set_ylabel("Number of clones")
axes[0].set_title("Clone size distribution (capped at 30)")
axes[0].axvline(x=3,  color="#E24B4A", linestyle="--", linewidth=1, label="min=3")
axes[0].axvline(x=5,  color="#EF9F27", linestyle="--", linewidth=1, label="min=5")
axes[0].axvline(x=10, color="#1D9E75", linestyle="--", linewidth=1, label="min=10")
axes[0].legend()

# Cumulative cells retained vs min_size
min_sizes   = range(1, 31)
cells_kept  = [clone_sizes[clone_sizes >= m].sum() for m in min_sizes]
pct_kept    = [c / clone_sizes.sum() * 100 for c in cells_kept]
axes[1].plot(list(min_sizes), pct_kept, color="#534AB7", linewidth=2)
axes[1].axvline(x=3,  color="#E24B4A", linestyle="--", linewidth=1, label="min=3")
axes[1].axvline(x=5,  color="#EF9F27", linestyle="--", linewidth=1, label="min=5")
axes[1].axvline(x=10, color="#1D9E75", linestyle="--", linewidth=1, label="min=10")
axes[1].set_xlabel("Minimum clone size")
axes[1].set_ylabel("% of barcoded cells retained")
axes[1].set_title("Cells retained vs minimum clone size")
axes[1].legend()
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(
    "/srv/johan/Course work/Biologically Informed Neural Networks/clone_size_distribution.png",
    dpi=150, bbox_inches="tight"
)
plt.show()

# ── Spatial noise check ───────────────────────────────────────────────────────
# For each clone, compute the spread of its cells in UMAP space
# High spread = spatially scattered = likely noisy
umap      = adata.obsm["X_umap_neural_42"]
barcoded  = adata[adata.obs["cloneid"] != "NA"].copy()
umap_bc   = barcoded.obsm["X_umap_neural_42"]

clone_spread = {}
for clone_id, group in barcoded.obs.groupby("cloneid"):
    idx    = barcoded.obs_names.get_indexer(group.index)
    coords = umap_bc[idx]
    if len(coords) >= 2:
        # Mean pairwise distance as spread metric
        from scipy.spatial.distance import pdist
        spread = pdist(coords).mean()
        clone_spread[clone_id] = {"spread": spread, "size": len(coords)}

spread_df = pd.DataFrame(spread_df).T if False else pd.DataFrame(clone_spread).T
spread_df = pd.DataFrame.from_dict(clone_spread, orient="index")
spread_df.columns = ["spread", "size"]
spread_df["size"]   = spread_df["size"].astype(int)
spread_df["spread"] = spread_df["spread"].astype(float)

print("\n── Spatial spread by clone size ─────────────────────────────────────")
print(f"  {'Size bin':>12s}  {'N clones':>10s}  {'Mean spread':>12s}  {'Median spread':>14s}")
for min_s, max_s in [(2,2),(3,4),(5,7),(8,14),(15,30),(31,999)]:
    mask = (spread_df["size"] >= min_s) & (spread_df["size"] <= max_s)
    if mask.sum() > 0:
        print(f"  {min_s:>4d}–{max_s:<6d}    {mask.sum():>10d}  "
              f"{spread_df.loc[mask,'spread'].mean():>12.3f}  "
              f"{spread_df.loc[mask,'spread'].median():>14.3f}")

# ── What spread threshold removes the noisiest clones? ───────────────────────
spread_pcts = np.percentile(spread_df["spread"], [50, 75, 90, 95])
print(f"\n  Spread percentiles:")
for p, v in zip([50, 75, 90, 95], spread_pcts):
    print(f"    {p}th percentile: {v:.3f}")

#Noise spread NOT included yet

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BINN Preprocessing Notebook — v2                                            ║
# ║  Biologically Informed Neural Network for Lineage Prediction                 ║
# ║                                                                              ║
# ║  USAGE: Edit the CONFIG section below, then run all cells end-to-end.        ║
# ║  Run this notebook every time you change any CONFIG parameter.               ║
# ║                                                                              ║
# ║  OUTPUT files (all written to SAVE_DIR):                                     ║
# ║    adata_preprocessed.h5ad  — AnnData with all modality scores + labels      ║
# ║    label_encoders.pkl        — LabelEncoders for lineage + celltype          ║
# ║    dims.json                 — Input dimensions for BINN model               ║
# ║    go_net.csv                — GO network used for AUCell scoring            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — edit this section only
# All dataset-specific variables live here. Nothing below needs to change.
# ══════════════════════════════════════════════════════════════════════════════

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH  = "/srv/johan/Python/E9 E10 Data/E9E10 Neural/E9E10_CNS_ONLY.h5ad"
SAVE_DIR   = "/srv/johan/Course work/Biologically Informed Neural Networks"

# ── Clone2vec parameters ──────────────────────────────────────────────────────
MIN_CLONE_SIZE  = 5          # Minimum cells per clone to keep for training
                             # Higher = cleaner labels, fewer training cells
                             # Recommended range: 3–20
C2V_RESOLUTION  = 3.0        # Leiden resolution for clonal cluster granularity
                             # Higher = more clusters
C2V_BATCH_SIZE  = 4096       # clone2vec training batch size
C2V_PATIENCE    = 3          # clone2vec early stopping patience

# ── Column names — adjust to match your dataset's obs columns ────────────────
CLONEID_COL     = "cloneid"          # Column holding clone/barcode IDs
                                     # Expected: integers or strings, NA for unbarcoded
CLONEID_NA_INT  = -2147483648        # R NA_integer_ sentinel value (if applicable)
CLUSTER_KEY     = "SCT_snn_res.2"    # Transcriptomic cluster for clone2vec context
CELLTYPE_KEY    = "SCT_snn_res.2"    # Celltype label for adversarial head
                                     # Can be same as CLUSTER_KEY or a richer annotation
                                     # e.g. "majority_voting", "predicted.celltype"
UMAP_KEY        = "X_umap_neural_42" # UMAP embedding key in adata.obsm for plotting
PCA_KEY         = "X_pca_neural"     # PCA/dimred key in adata.obsm for clonal_nn

# ── Biological parameters ─────────────────────────────────────────────────────
ORGANISM        = "mouse"            # "mouse" or "human"
TF_CONFIDENCE   = ["A", "B", "C"]   # DoRothEA confidence levels to include
PROGENY_TOP     = 500                # Number of top genes per PROGENy pathway
GO_LIBRARY      = "GO_Biological_Process_2023"
GO_ORGANISM     = "Mouse"            # "Mouse" or "Human" (capitalised for gseapy)
GO_MIN_SIZE     = 10                 # Min genes per GO term
GO_MAX_SIZE     = 500                # Max genes per GO term

# ── Dataset split ─────────────────────────────────────────────────────────────
VAL_FRACTION    = 0.15               # Fraction of barcoded cells held out for validation
RANDOM_SEED     = 42                 # Reproducibility seed

# ══════════════════════════════════════════════════════════════════════════════
# END CONFIG — do not edit below this line
# ══════════════════════════════════════════════════════════════════════════════


import os
import sys
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import sclitr as sl
import decoupler as dc
import gseapy as gp
import torch
from torch.utils.data import Dataset, random_split
from scipy.sparse import issparse
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings("ignore")

sc.settings.verbosity = 3

print("╔══════════════════════════════════════════════════════════════════╗")
print("║  BINN Preprocessing v2                                           ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print()
print("── Configuration ────────────────────────────────────────────────────")
print(f"  DATA_PATH       : {DATA_PATH}")
print(f"  SAVE_DIR        : {SAVE_DIR}")
print(f"  MIN_CLONE_SIZE  : {MIN_CLONE_SIZE}")
print(f"  C2V_RESOLUTION  : {C2V_RESOLUTION}")
print(f"  CLONEID_COL     : {CLONEID_COL}")
print(f"  CLUSTER_KEY     : {CLUSTER_KEY}")
print(f"  CELLTYPE_KEY    : {CELLTYPE_KEY}")
print(f"  UMAP_KEY        : {UMAP_KEY}")
print(f"  PCA_KEY         : {PCA_KEY}")
print(f"  ORGANISM        : {ORGANISM}")
print(f"  VAL_FRACTION    : {VAL_FRACTION}")
print("─────────────────────────────────────────────────────────────────────")


# ══════════════════════════════════════════════════════════════════════════════
# Step 1 — Load data
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 1: Load data ────────────────────────────────────────────────")

adata = sc.read_h5ad(DATA_PATH)
adata.var_names_make_unique()

# ── Validate required columns exist ──────────────────────────────────────────
missing_cols = []
for col in [CLONEID_COL, CLUSTER_KEY, CELLTYPE_KEY]:
    if col not in adata.obs.columns:
        missing_cols.append(col)
if missing_cols:
    raise ValueError(
        f"Missing obs columns: {missing_cols}\n"
        f"Available columns: {adata.obs.columns.tolist()}\n"
        f"Update CONFIG to use correct column names."
    )

missing_obsm = []
for key in [UMAP_KEY, PCA_KEY]:
    if key not in adata.obsm:
        missing_obsm.append(key)
if missing_obsm:
    raise ValueError(
        f"Missing obsm keys: {missing_obsm}\n"
        f"Available obsm keys: {list(adata.obsm.keys())}\n"
        f"Update CONFIG to use correct obsm key names."
    )

print(f"  Validation passed — all required columns and obsm keys found.")

# ── Convert cloneid to clean string — handles all formats ────────────────────
# Supports: integer with R NA sentinel, string, categorical
cloneid = adata.obs[CLONEID_COL]
if cloneid.dtype.name in ["int32", "int64", "Int32", "Int64"]:
    # Fresh from R — replace integer sentinel with string NA
    adata.obs[CLONEID_COL] = (
        cloneid
        .replace(CLONEID_NA_INT, pd.NA)
        .astype("Int64")
        .astype(str)
        .replace("<NA>", "NA")
    )
else:
    # Already string or category — normalise all NA variants
    adata.obs[CLONEID_COL] = (
        cloneid
        .astype(str)
        .replace(str(CLONEID_NA_INT), "NA")
        .replace("nan",  "NA")
        .replace("<NA>", "NA")
        .replace("None", "NA")
    )

n_barcoded   = (adata.obs[CLONEID_COL] != "NA").sum()
n_unbarcoded = (adata.obs[CLONEID_COL] == "NA").sum()

print(f"  {CLONEID_COL} dtype : {adata.obs[CLONEID_COL].dtype}")
print(f"  Total cells        : {adata.n_obs:,}")
print(f"  Barcoded cells     : {n_barcoded:,}")
print(f"  Unbarcoded cells   : {n_unbarcoded:,}")
print(adata)


# ══════════════════════════════════════════════════════════════════════════════
# Step 2 — Clone2vec
# Produces c2v_leiden labels on adata.obs
# Only clones >= MIN_CLONE_SIZE are used
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n── Step 2: Clone2vec (min_size={MIN_CLONE_SIZE}, resolution={C2V_RESOLUTION}) ──")

clones = sl.pp.clones_adata(
    adata,
    obs_name=CLONEID_COL,
    fill_obs=CLUSTER_KEY,
    min_size=MIN_CLONE_SIZE,
    na_value="NA",
)

sl.tl.clonal_nn(adata, clones, use_rep=PCA_KEY)

sl.tl.clone2vec(
    clones,
    batch_size=C2V_BATCH_SIZE,
    early_stopping_patience=C2V_PATIENCE,
)

sl.pl.loss_history(clones)

sc.pp.neighbors(clones, use_rep="clone2vec")
sc.tl.umap(clones)
sc.tl.leiden(
    clones,
    resolution=C2V_RESOLUTION,
    flavor="igraph",
    n_iterations=2,
)

sl.pp.transfer_annotation(
    adata, clones,
    annotation_obs_clones="leiden",
    obs_name=CLONEID_COL,
    na_value="NA",
)

n_clonal_clusters = adata.obs["c2v_leiden"].nunique()
print(f"  Clones selected   : {clones.n_obs:,}")
print(f"  Clonal clusters   : {n_clonal_clusters}")
print(adata.obs["c2v_leiden"].value_counts())

# ── Clone size summary ────────────────────────────────────────────────────────
barcoded_mask    = adata.obs["c2v_leiden"] != "NA"
barcoded_obs     = adata.obs.loc[barcoded_mask].copy()
cells_per_clone  = barcoded_obs.groupby(CLONEID_COL, observed=True).size()
cells_per_clone  = cells_per_clone[cells_per_clone > 0]

print(f"\n  Clone size (retained clones):")
print(f"    min={cells_per_clone.min()}  max={cells_per_clone.max()}  "
      f"mean={cells_per_clone.mean():.1f}  n_clones={len(cells_per_clone)}")

# ── Visualise clone2vec ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.embedding(
    adata, basis=UMAP_KEY,
    color="c2v_leiden",
    title="Clone2vec clonal clusters",
    frameon=False, legend_loc="on data",
    legend_fontsize=5, ax=axes[0], show=False,
)
sc.pl.embedding(
    adata, basis=UMAP_KEY,
    color=CLUSTER_KEY,
    title=f"Transcriptomic clusters ({CLUSTER_KEY})",
    frameon=False, legend_loc="on data",
    legend_fontsize=5, ax=axes[1], show=False,
)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "c2v_clusters.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"  Saved: c2v_clusters.png")


# ══════════════════════════════════════════════════════════════════════════════
# Step 3a — TF activity (DoRothEA via decoupleR)
# Stored in adata.obsm["X_tf"]
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n── Step 3a: TF activity (DoRothEA, organism={ORGANISM}) ─────────────")

net_tf = dc.op.dorothea(organism=ORGANISM, levels=TF_CONFIDENCE)
print(f"  Network size: {len(net_tf)} edges, {net_tf['source'].nunique()} TFs")

dc.mt.ulm(adata, net=net_tf, verbose=True)
adata.obsm["X_tf"]    = adata.obsm["score_ulm"].values
adata.uns["tf_names"] = list(adata.obsm["score_ulm"].columns)

print(f"  TFs scored: {adata.obsm['X_tf'].shape[1]}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 3b — Pathway scores (PROGENy via decoupleR)
# Stored in adata.obsm["X_pathway"]
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n── Step 3b: Pathway scores (PROGENy top={PROGENY_TOP}) ──────────────")

net_pathway = dc.op.progeny(organism=ORGANISM, top=PROGENY_TOP)
print(f"  Network size: {len(net_pathway)} edges")

dc.mt.mlm(adata, net=net_pathway, verbose=True)
adata.obsm["X_pathway"]    = adata.obsm["score_mlm"].values
adata.uns["pathway_names"] = list(adata.obsm["score_mlm"].columns)

print(f"  Pathways scored: {adata.obsm['X_pathway'].shape[1]}")
print(f"  Pathway names  : {adata.uns['pathway_names']}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 3c — GO term scores (AUCell via decoupleR)
# Stored in adata.obsm["X_go"]
# Gene names converted to title case to match mouse gene naming (e.g. Pax6)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n── Step 3c: GO term scores (AUCell, {GO_LIBRARY}) ───────────────────")

gene_sets_raw = gp.get_library(GO_LIBRARY, organism=GO_ORGANISM)
gene_sets     = {k: v for k, v in gene_sets_raw.items()
                 if GO_MIN_SIZE <= len(v) <= GO_MAX_SIZE}
print(f"  GO terms after size filtering ({GO_MIN_SIZE}–{GO_MAX_SIZE}): {len(gene_sets)}")

# Convert gene names to title case to match dataset gene naming convention
# e.g. HSPA9 (human) → Hspa9 (mouse)
go_rows = []
for term, genes in gene_sets.items():
    for g in genes:
        g_conv = g[0].upper() + g[1:].lower()
        if g_conv in adata.var_names:
            go_rows.append({"source": term, "target": g_conv, "weight": 1.0})
go_net = pd.DataFrame(go_rows)

overlap = go_net["target"].nunique()
print(f"  Gene overlap with dataset: {overlap} / {adata.n_vars} genes "
      f"({overlap/adata.n_vars*100:.1f}%)")
print(f"  GO network edges: {len(go_net)}")

dc.mt.aucell(adata, net=go_net, verbose=True)
adata.obsm["X_go"]    = adata.obsm["score_aucell"].values
adata.uns["go_names"] = list(adata.obsm["score_aucell"].columns)

print(f"  GO terms scored: {adata.obsm['X_go'].shape[1]}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 4 — Encode labels
# lineage_code : c2v_leiden → integer (−1 for unbarcoded)
# celltype_code: CELLTYPE_KEY → integer (all cells)
# Must rerun every time clone2vec produces new labels
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 4: Encode labels ────────────────────────────────────────────")

barcoded_mask = adata.obs["c2v_leiden"] != "NA"

le_lineage  = LabelEncoder()
le_celltype = LabelEncoder()

# Lineage codes — only barcoded cells, −1 for unbarcoded
adata.obs["lineage_code"] = -1
adata.obs.loc[barcoded_mask, "lineage_code"] = le_lineage.fit_transform(
    adata.obs.loc[barcoded_mask, "c2v_leiden"].astype(str)
)

# Celltype codes — all cells (used for adversarial disentanglement only)
adata.obs["celltype_code"] = le_celltype.fit_transform(
    adata.obs[CELLTYPE_KEY].astype(str)
)

n_lineage_classes  = adata.obs["lineage_code"].nunique() - 1  # -1 excludes the NA class
n_celltype_classes = adata.obs["celltype_code"].nunique()

print(f"  Lineage classes  : {n_lineage_classes}")
print(f"  Cell type classes: {n_celltype_classes}")
print(f"  Barcoded cells   : {barcoded_mask.sum():,}")
print(f"  Unbarcoded cells : {(~barcoded_mask).sum():,}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 5 — Build PyTorch Dataset and compute dims
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 5: Build dataset ────────────────────────────────────────────")

class BINNDataset(Dataset):
    """
    PyTorch Dataset for BINN training.
    Returns per-cell multimodal feature vectors and labels.
    barcoded_only=True: only cells with lineage_code >= 0 (training set)
    barcoded_only=False: all cells (inference set)
    """
    def __init__(self, adata, barcoded_only=True):
        if barcoded_only:
            mask  = adata.obs["lineage_code"] >= 0
            adata = adata[mask].copy()

        X = adata.X.toarray() if issparse(adata.X) else np.array(adata.X)

        self.expr     = torch.tensor(X,                                  dtype=torch.float32)
        self.tf       = torch.tensor(adata.obsm["X_tf"],                 dtype=torch.float32)
        self.pathway  = torch.tensor(adata.obsm["X_pathway"],            dtype=torch.float32)
        self.go       = torch.tensor(adata.obsm["X_go"],                 dtype=torch.float32)
        self.lineage  = torch.tensor(adata.obs["lineage_code"].values,   dtype=torch.long)
        self.celltype = torch.tensor(adata.obs["celltype_code"].values,  dtype=torch.long)

        self.n_genes     = self.expr.shape[1]
        self.n_tfs       = self.tf.shape[1]
        self.n_pathways  = self.pathway.shape[1]
        self.n_go        = self.go.shape[1]
        self.n_lineages  = int(self.lineage.max().item()) + 1
        self.n_celltypes = int(self.celltype.max().item()) + 1

    def __len__(self):
        return self.expr.shape[0]

    def __getitem__(self, idx):
        return {
            "expr":     self.expr[idx],
            "tf":       self.tf[idx],
            "pathway":  self.pathway[idx],
            "go":       self.go[idx],
            "lineage":  self.lineage[idx],
            "celltype": self.celltype[idx],
            "cell_idx": idx,
        }

# Training dataset — barcoded cells only
dataset_full = BINNDataset(adata, barcoded_only=True)
n_val        = int(len(dataset_full) * VAL_FRACTION)
n_train      = len(dataset_full) - n_val

dataset_train, dataset_val = random_split(
    dataset_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

# Inference dataset — all cells including unbarcoded
dataset_infer = BINNDataset(adata, barcoded_only=False)

dims = {
    "n_genes":        dataset_full.n_genes,
    "n_tfs":          dataset_full.n_tfs,
    "n_pathways":     dataset_full.n_pathways,
    "n_go":           dataset_full.n_go,
    "n_lineages":     dataset_full.n_lineages,
    "n_celltypes":    dataset_full.n_celltypes,
    "min_clone_size": int(MIN_CLONE_SIZE),
    "c2v_resolution": float(C2V_RESOLUTION),
    "val_fraction":   float(VAL_FRACTION),
    "random_seed":    int(RANDOM_SEED),
}

print(f"  Train    : {n_train:,} cells")
print(f"  Val      : {n_val:,} cells")
print(f"  Inference: {len(dataset_infer):,} cells")
print(f"  Dims     : {dims}")

# ── Sanity check batch ────────────────────────────────────────────────────────
from torch.utils.data import DataLoader
loader = DataLoader(dataset_train, batch_size=64, shuffle=True)
batch  = next(iter(loader))
print("\n  Batch sanity check:")
for k, v in batch.items():
    print(f"    {k:12s}  shape={str(v.shape):30s}  dtype={v.dtype}")


# ══════════════════════════════════════════════════════════════════════════════
# Step 6 — Save all outputs
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Step 6: Save ─────────────────────────────────────────────────────")

os.makedirs(SAVE_DIR, exist_ok=True)

# Remove any non-h5ad-serializable objects from uns before saving
for key in ["le_lineage", "le_celltype", "clonal_nn", "clone2vec"]:
    adata.uns.pop(key, None)
adata.raw = None

# Save AnnData with all modality scores and labels
adata.write_h5ad(os.path.join(SAVE_DIR, "adata_preprocessed.h5ad"))
print(f"  Saved: adata_preprocessed.h5ad")

# Save label encoders (needed for inverse_transform at inference time)
with open(os.path.join(SAVE_DIR, "label_encoders.pkl"), "wb") as f:
    pickle.dump({"le_lineage": le_lineage, "le_celltype": le_celltype}, f)
print(f"  Saved: label_encoders.pkl")

# Save dims — convert numpy types to native Python for JSON serialisation
dims_serializable = {
    k: float(v) if isinstance(v, float) else int(v)
    for k, v in dims.items()
}
with open(os.path.join(SAVE_DIR, "dims.json"), "w") as f:
    json.dump(dims_serializable, f, indent=2)
print(f"  Saved: dims.json")

# Save GO network (needed to score new datasets with identical feature space)
go_net.to_csv(os.path.join(SAVE_DIR, "go_net.csv"), index=False)
print(f"  Saved: go_net.csv")


# ══════════════════════════════════════════════════════════════════════════════
# Step 7 — Final summary
# ══════════════════════════════════════════════════════════════════════════════
barcoded_obs    = adata.obs.loc[barcoded_mask].copy()
cells_per_clone = barcoded_obs.groupby(CLONEID_COL, observed=True).size()
cells_per_clone = cells_per_clone[cells_per_clone > 0]

print("\n╔══════════════════════════════════════════════════════════════════╗")
print("║  Preprocessing complete                                           ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(f"  Dataset          : {os.path.basename(DATA_PATH)}")
print(f"  min_clone_size   : {MIN_CLONE_SIZE}")
print(f"  c2v_resolution   : {C2V_RESOLUTION}")
print(f"  Clones retained  : {len(cells_per_clone):,}")
print(f"  Clone size       : min={cells_per_clone.min()}  "
      f"max={cells_per_clone.max()}  mean={cells_per_clone.mean():.1f}")
print(f"  Barcoded cells   : {barcoded_mask.sum():,}")
print(f"  Lineage classes  : {dims['n_lineages']}")
print(f"  Cell type classes: {dims['n_celltypes']}")
print(f"  Train / Val      : {n_train:,} / {n_val:,}")
print(f"  Saved to         : {SAVE_DIR}")
print()
print("  → Now run BINN_Training notebook.")